In [ ]:
import pandas as pd
import ibis

from swed_17.zone_db import CBRFCZone

In [ ]:
m3w_swe = pd.read_csv("/nvm9/data/m3works/cbrfc_zonal_wy2026.csv")[["date", "zone", "mean_swe_in"]]

In [ ]:
m3w_swe.head()

In [ ]:
m3w_swe.count()

In [ ]:
DB_CONNECTION = "service=swe_db"
SWE_DB = ibis.connect("postgres://?" + DB_CONNECTION)
zone_db = CBRFCZone(DB_CONNECTION)

In [ ]:
DB_TABLE = "m3w_zonal_swe"
zones_in_isnobal = SWE_DB.table("cbrfc_zones_in_isnobal")

In [ ]:
zone_info = zones_in_isnobal.select(zones_in_isnobal.gid, zones_in_isnobal.zone).distinct().execute()

In [ ]:
zone_info.head()

In [ ]:
m3w_swe = pd.merge(m3w_swe, zone_info, on="zone", how="left")
m3w_swe.drop("zone", axis=1, inplace=True)

In [ ]:
m3w_swe.head()

In [ ]:
m3w_swe.rename(columns={
    "gid": "cbrfc_zone_id",
    "mean_swe_in": "value",
    "date": "datetime",
}, inplace=True)

m3w_swe["datetime"] = pd.to_datetime(
    m3w_swe["datetime"]
).dt.tz_localize("UTC")
m3w_swe["metric_type_id"] = 1
m3w_swe["value"] *= 25.4

In [ ]:
m3w_swe.head()

In [ ]:
m3w_swe.count()

In [ ]:
m3w_swe[m3w_swe.duplicated(subset=['datetime', 'cbrfc_zone_id'])]

In [ ]:
zone_db.write(m3w_swe, DB_TABLE)